# Notebook 02 · Red de una capa y red multicapa con Sigmoide, ReLU y *backpropagation* (NumPy)

| R1-A2-S4 · Redes Neuronales II | 604013/DEEP LEARNING -CONCEPTOS/FU/CAD2202023205/EIAIIPA2026 |
|---|---|
| Universidad de Cundinamarca · Especialización en Inteligencia Artificial | Docente: **Nidia Stella Garcia Roa** |
| Estudiante: **Yohann Javier Nikolay Salazar Cardenas** | Repositorio: `https://github.com/Yohann0011/Redes-Neuronales-II` |


**Problema:** el mismo dataset de frutas del Notebook 01 (clasificación **binaria**: 0 = manzana,
1 = banano), pero ahora con redes que **aprenden minimizando una función de pérdida** mediante
descenso de gradiente y *backpropagation*.

> **Alcance:** (a) red neuronal de **una capa** con activación Sigmoide y pérdida *Binary Cross-Entropy*;
> (b) red **multicapa** 3 → 4 → 1 con ReLU en la capa oculta e inicialización de He;
> (c) comparación experimental ReLU vs Sigmoide en capas ocultas (desvanecimiento del gradiente).

---

## 1. Del perceptrón a la red que aprende de su error

| Redes Neuronales I / Notebook 01 | Redes Neuronales II / este notebook |
|---|---|
| Pesos definidos manualmente o por regla de corrección | Pesos ajustados minimizando una función de pérdida |
| Solo *forward propagation* | *Forward* + *loss* + *backpropagation* + actualización |
| Activación escalón (no derivable) | Sigmoide y ReLU (deribles casi en todo punto) |
| Salida discreta | Salida probabilística en [0, 1] |
| La red calcula | La red aprende |

## 2. Fundamento matemático

**Sigmoide** y su derivada (propiedad que la hace cómoda para *backpropagation*):

$$ \sigma(z)=\frac{1}{1+e^{-z}}
\qquad\qquad
\sigma'(z)=\sigma(z)\,(1-\sigma(z)) $$

**ReLU** y su derivada:

$$ \mathrm{ReLU}(z)=\max(0,z)
\qquad\qquad
\mathrm{ReLU}'(z)=\begin{cases}1 & z>0\\ 0 & z\le 0\end{cases} $$

**Binary Cross-Entropy** (pérdida para dos clases), con $m$ ejemplos:

$$ L=-\frac{1}{m}\sum_{k=1}^{m}\left[y^{(k)}\log\hat y^{(k)}+(1-y^{(k)})\log(1-\hat y^{(k)})\right] $$

**Backpropagation** es la aplicación sistemática de la regla de la cadena para obtener
$\partial L/\partial w$ en cada capa. Para la red de una capa con Sigmoide el resultado se simplifica a:

$$ \frac{\partial L}{\partial z}=\hat y-y
\qquad
\frac{\partial L}{\partial \mathbf w}=\frac{1}{m}X^\top(\hat y-y)
\qquad
\frac{\partial L}{\partial b}=\frac{1}{m}\sum(\hat y-y) $$

y la actualización por descenso de gradiente es $\mathbf w \leftarrow \mathbf w-\eta\,\partial L/\partial\mathbf w$.

Para la red **multicapa** la señal de error se propaga hacia atrás:

$$ \delta_2=A_2-y \;\rightarrow\;
\frac{\partial L}{\partial W_2}=\frac{A_1^\top\delta_2}{m} \;\rightarrow\;
\delta_1=(\delta_2 W_2^\top)\odot \mathrm{ReLU}'(Z_1) \;\rightarrow\;
\frac{\partial L}{\partial W_1}=\frac{X^\top\delta_1}{m} $$


## 3. Entorno y datos

Se reutiliza el dataset, la partición y el escalado del Notebook 01 para que la comparación entre
modelos sea justa (mismas frutas de entrenamiento y mismas frutas de prueba).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 80,
    "savefig.dpi": 80,
    "savefig.bbox": None,
    "figure.autolayout": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 9,
})
plt.rcParams["font.family"] = "DejaVu Sans"

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import random
random.seed(42)
np.random.seed(42)
print("NumPy:", np.__version__, "| entorno listo")

NumPy: 2.4.6 | entorno listo


In [2]:
X = np.array([
    # MANZANAS  (clase 0): [peso g, longitud cm, nivel de amarillo 0-1]
    [150, 7.5, 0.10], [160, 8.0, 0.15], [145, 7.2, 0.05],
    [170, 8.3, 0.20], [155, 7.8, 0.10], [165, 8.1, 0.15],
    [140, 7.0, 0.05], [175, 8.5, 0.20], [158, 7.9, 0.10],
    [162, 8.2, 0.15],

    # BANANOS   (clase 1)
    [120, 18.0, 0.90], [125, 19.0, 0.95], [115, 17.5, 0.85],
    [130, 20.0, 1.00], [118, 18.5, 0.90], [123, 19.5, 0.95],
    [110, 17.0, 0.80], [128, 20.5, 1.00], [121, 18.8, 0.90],
    [126, 19.2, 0.95]
], dtype=float)

y = np.array([0] * 10 + [1] * 10)

print("Matriz de características X:", X.shape, "-> (20 frutas, 3 características)")
print("Vector de etiquetas   y:", y.shape, "-> (0 = manzana, 1 = banano)")
print()
print("Primeras 3 manzanas:\n", X[:3])
print("Primeros 3 bananos:\n", X[10:13])


Matriz de características X: (20, 3) -> (20 frutas, 3 características)
Vector de etiquetas   y: (20,) -> (0 = manzana, 1 = banano)

Primeras 3 manzanas:
 [[1.50e+02 7.50e+00 1.00e-01]
 [1.60e+02 8.00e+00 1.50e-01]
 [1.45e+02 7.20e+00 5.00e-02]]
Primeros 3 bananos:
 [[120.    18.     0.9 ]
 [125.    19.     0.95]
 [115.    17.5    0.85]]


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 80 % entrenamiento / 20 % prueba, conservando la proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# El escalado se AJUSTA solo con entrenamiento y se APLICA a prueba
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)
print("Distribución de clases en prueba:", np.bincount(y_test))
print()
print("Media de cada característica en entrenamiento (debe ser ~0):")
print(np.round(X_train.mean(axis=0), 6))
print("Desviación estándar (debe ser ~1):")
print(np.round(X_train.std(axis=0), 6))


X_train: (16, 3) | X_test: (4, 3)
y_train: (16,) | y_test: (4,)
Distribución de clases en prueba: [2 2]

Media de cada característica en entrenamiento (debe ser ~0):
[0. 0. 0.]
Desviación estándar (debe ser ~1):
[1. 1. 1.]


## 4. Funciones de activación y función de pérdida

Se implementan Sigmoide, ReLU, la derivada de ReLU y la *Binary Cross-Entropy*. El término `epsilon`
evita el logaritmo de cero (inestabilidad numérica) cuando la red está muy segura y se equivoca.

In [4]:
def sigmoid(z):
    """Función de activación Sigmoide: comprime z al intervalo (0, 1)."""
    return 1.0 / (1.0 + np.exp(-z))


def relu(z):
    """Función de activación ReLU: los negativos se anulan, los positivos pasan."""
    return np.maximum(0.0, z)


def derivada_relu(z):
    """Derivada de ReLU: 1 si z > 0, 0 en caso contrario."""
    return (z > 0).astype(float)


def calcular_error(y_real, y_predicha):
    """Binary Cross-Entropy promedio."""
    epsilon = 1e-8
    return -np.mean(
        y_real * np.log(y_predicha + epsilon)
        + (1 - y_real) * np.log(1 - y_predicha + epsilon)
    )


# --- Verificación numérica de las activaciones ---
z_prueba = np.array([-3.0, -1.0, 0.0, 1.0, 3.0])
print("z            :", z_prueba)
print("sigmoid(z)   :", np.round(sigmoid(z_prueba), 4))
print("relu(z)      :", relu(z_prueba))
print("relu'(z)     :", derivada_relu(z_prueba))
print()
print("Error si predice 0.5 para y=1 :", round(float(calcular_error(np.array([[1.0]]), np.array([[0.5]]))), 4))
print("Error si predice 0.99 para y=1:", round(float(calcular_error(np.array([[1.0]]), np.array([[0.99]]))), 4))
print("Error si predice 0.01 para y=1:", round(float(calcular_error(np.array([[1.0]]), np.array([[0.01]]))), 4))
print("-> La pérdida crece cuando la red está segura y se equivoca: por eso penaliza más los errores graves.")

z            : [-3. -1.  0.  1.  3.]
sigmoid(z)   : [0.0474 0.2689 0.5    0.7311 0.9526]
relu(z)      : [0. 0. 0. 1. 3.]
relu'(z)     : [0. 0. 0. 1. 1.]

Error si predice 0.5 para y=1 : 0.6931
Error si predice 0.99 para y=1: 0.0101
Error si predice 0.01 para y=1: 4.6052
-> La pérdida crece cuando la red está segura y se equivoca: por eso penaliza más los errores graves.


---

# 5. Red neuronal de UNA CAPA con Sigmoide (3 → 1)

Arquitectura: tres entradas, **una sola neurona** con activación Sigmoide y salida probabilística.
A diferencia del perceptrón, aquí los pesos se actualizan con el gradiente de la pérdida en **todas**
las épocas, no solo cuando hay error.

### 5.1 Inicialización y primera propagación hacia adelante

Los pesos se inicializan aleatoriamente con escala pequeña (0.01). Con pesos casi nulos, $z\approx 0$ y
$\sigma(0)=0.5$: la red arranca **sin opinión**, que es el comportamiento esperado antes de aprender.

In [5]:
np.random.seed(42)
W = np.random.randn(3, 1) * 0.01
b = 0.0

Z = X_train.reshape(-1, 3) @ W + b          # combinación lineal
A = sigmoid(Z)                              # activación

y_col = y_train.reshape(-1, 1)              # etiquetas en forma de columna

print("Pesos iniciales W =", np.round(W.ravel(), 6), "| b =", round(b, 6))
print()
print("Primeras 5 predicciones (probabilidades):", np.round(A.ravel()[:5], 4))
print("Primeras 5 etiquetas reales             :", y_train[:5])
print()
print("Error inicial (BCE):", round(float(calcular_error(y_col, A)), 4))
print("-> Un valor cercano a ln(2) = 0.6931 indica que la red predice ~0.5 para todo: aún no aprende.")

Pesos iniciales W = [ 0.004967 -0.001383  0.006477] | b = 0.0

Primeras 5 predicciones (probabilidades): [0.5002 0.5004 0.4998 0.4996 0.5002]
Primeras 5 etiquetas reales             : [0 0 1 1 1]

Error inicial (BCE): 0.6929
-> Un valor cercano a ln(2) = 0.6931 indica que la red predice ~0.5 para todo: aún no aprende.


### 5.2 Una vuelta de *backpropagation*, paso a paso

Se ejecuta **un solo paso** de gradiente imprimiendo cada cantidad intermedia. `dW` y `db` no son "el
error": son las direcciones en las que deben moverse los parámetros para reducir la pérdida.

In [6]:
m = len(X_train)

# 1) Forward
Z = X_train @ W + b
A = sigmoid(Z)

# 2) Loss
error = calcular_error(y_col, A)

# 3) Backpropagation (regla de la cadena)
dZ = A - y_col                     # dL/dZ  para BCE + Sigmoide
dW = X_train.T @ dZ / m            # dL/dW
db = np.mean(dZ)                   # dL/db

# 4) Descenso de gradiente
learning_rate = 0.1
W_nuevo = W - learning_rate * dW
b_nuevo = b - learning_rate * db

print("Pérdida antes de actualizar      :", round(float(error), 6))
print("dZ (primeros 5)                  :", np.round(dZ.ravel()[:5], 5))
print("dW = X^T dZ / m                  :", np.round(dW.ravel(), 6))
print("db                               :", round(float(db), 6))
print()
print("W antes:", np.round(W.ravel(), 6), "-> W después:", np.round(W_nuevo.ravel(), 6))
print("b antes:", round(float(b), 6), "        -> b después:", round(float(b_nuevo), 6))

# Verificación: la pérdida debe disminuir tras el paso
A2 = sigmoid(X_train @ W_nuevo + b_nuevo)
print()
print("Pérdida después de actualizar    :", round(float(calcular_error(y_col, A2)), 6))
print("¿Disminuyó?", calcular_error(y_col, A2) < error)

Pérdida antes de actualizar      : 0.692932
dZ (primeros 5)                  : [ 0.50019  0.50039 -0.50025 -0.50041 -0.49978]
dW = X^T dZ / m                  : [ 0.466997 -0.497071 -0.497324]
db                               : 0.0

W antes: [ 0.004967 -0.001383  0.006477] -> W después: [-0.041733  0.048325  0.056209]
b antes: 0.0         -> b después: -0.0

Pérdida después de actualizar    : 0.624232
¿Disminuyó? True


### 5.3 Entrenamiento completo (2000 épocas)

Se registra la pérdida en cada época para graficar la **curva de aprendizaje**. El comportamiento
esperado es una caída rápida seguida de un aplanamiento (convergencia).

In [7]:
np.random.seed(42)
W = np.random.randn(3, 1) * 0.01
b = 0.0
learning_rate = 0.1
epochs = 2000

historial_1capa = []
y_col = y_train.reshape(-1, 1)

for epoch in range(epochs):
    # 1. Forward propagation
    Z = X_train @ W + b
    A = sigmoid(Z)

    # 2. Loss
    error = float(calcular_error(y_col, A))
    historial_1capa.append(error)

    # 3. Backpropagation
    m = len(X_train)
    dZ = A - y_col
    dW = X_train.T @ dZ / m
    db = np.mean(dZ)

    # 4. Actualización de parámetros
    W = W - learning_rate * dW
    b = b - learning_rate * db

    if epoch % 400 == 0:
        print(f"Época {epoch:>5} | Loss {error:.6f}")

print(f"Época {epochs-1:>5} | Loss {historial_1capa[-1]:.6f}")
print()
print("Pesos finales W =", np.round(W.ravel(), 4))
print("Bias final    b =", round(float(b), 4))

W_1capa, b_1capa = W.copy(), float(b)

Época     0 | Loss 0.692932
Época   400 | Loss 0.009274
Época   800 | Loss 0.004618
Época  1200 | Loss 0.003075
Época  1600 | Loss 0.002306
Época  1999 | Loss 0.001846

Pesos finales W = [-1.9852  2.2612  2.2839]
Bias final    b = -0.0272


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

axes[0].plot(historial_1capa, color="#2980b9", lw=2)
axes[0].set_title("Figura 4. Curva de pérdida - red de una capa")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("Binary Cross-Entropy")
axes[0].set_yscale("log")

axes[1].plot(historial_1capa[:200], color="#c0392b", lw=2)
axes[1].set_title("Detalle de las primeras 200 épocas (escala lineal)")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Loss")

plt.tight_layout(); plt.savefig("fig03_loss_una_capa.png", bbox_inches="tight"); plt.show()

print("Pérdida inicial:", round(historial_1capa[0], 6))
print("Pérdida final  :", round(historial_1capa[-1], 6))
print("Reducción      :", f"{(1 - historial_1capa[-1]/historial_1capa[0])*100:.2f} %")

Pérdida inicial: 0.692932
Pérdida final  : 0.001846
Reducción      : 99.73 %


### 5.4 Evaluación de la red de una capa

Además del *accuracy* se muestran las **probabilidades**: esta es la ventaja práctica frente al
perceptrón, pues permite fijar umbrales de decisión según el costo de cada tipo de error.

In [9]:
probabilidades = sigmoid(X_test @ W_1capa + b_1capa)
predicciones = (probabilidades >= 0.5).astype(int).ravel()
acc_1capa = accuracy_score(y_test, predicciones)

print("=== RED DE UNA CAPA (Sigmoide + BCE + backpropagation) ===")
print(f"{'Fruta':>6} {'P(banano)':>10} {'Predicción':>12} {'Real':>12} {'Resultado':>10}")
for i in range(len(X_test)):
    pred = "BANANO" if predicciones[i] == 1 else "MANZANA"
    real = "BANANO" if y_test[i] == 1 else "MANZANA"
    ok = "ACIERTO" if pred == real else "ERROR"
    print(f"{i+1:>6} {probabilidades[i,0]:>10.4f} {pred:>12} {real:>12} {ok:>10}")

print()
print("Accuracy en prueba:", round(acc_1capa, 4))
print("Matriz de confusión:\n", confusion_matrix(y_test, predicciones))

=== RED DE UNA CAPA (Sigmoide + BCE + backpropagation) ===
 Fruta  P(banano)   Predicción         Real  Resultado
     1     0.0056      MANZANA      MANZANA    ACIERTO
     2     0.0009      MANZANA      MANZANA    ACIERTO
     3     0.9978       BANANO       BANANO    ACIERTO
     4     0.9979       BANANO       BANANO    ACIERTO

Accuracy en prueba: 1.0
Matriz de confusión:
 [[2 0]
 [0 2]]


---

# 6. Red neuronal MULTICAPA (3 → 4 → 1) con ReLU

Se reemplaza el "único juez" por un pequeño equipo: **cuatro neuronas ocultas** con ReLU analizan
distintas combinaciones de las características y **una neurona de salida** con Sigmoide produce la
probabilidad final.

```
  x1 ─┐
  x2 ─┼──► [ h1 h2 h3 h4 ]  ReLU ──► [ y ]  Sigmoide ──► P(banano)
  x3 ─┘        (capa oculta)            (salida)
     W1 (3×4)  b1 (1×4)             W2 (4×1)  b2 (1×1)
```

**Lectura de dimensiones:** `W1` tiene forma (3, 4) porque conecta 3 entradas con 4 neuronas ocultas;
`W2` tiene forma (4, 1) porque conecta 4 neuronas ocultas con 1 salida.

**Inicialización de He:** $W\sim\mathcal N\!\left(0,\sqrt{{2/n_{{ent}}}}\right)$ preserva la varianza de la
señal a través de capas ReLU y evita que los gradientes exploten o se desvanezcan desde la época 0.

In [10]:
def inicializar_parametros(entrada, ocultas, salida, semilla=42):
    """Inicialización de He para capas con ReLU."""
    np.random.seed(semilla)
    W1 = np.random.randn(entrada, ocultas) * np.sqrt(2.0 / entrada)
    b1 = np.zeros((1, ocultas))
    W2 = np.random.randn(ocultas, salida) * np.sqrt(2.0 / ocultas)
    b2 = np.zeros((1, salida))
    return W1, b1, W2, b2


W1, b1, W2, b2 = inicializar_parametros(3, 4, 1)

print("W1:", W1.shape, "-> conecta 3 entradas con 4 neuronas ocultas")
print("b1:", b1.shape)
print("W2:", W2.shape, "-> conecta 4 neuronas ocultas con 1 salida")
print("b2:", b2.shape)
print()
print("W1 =\n", np.round(W1, 4))
print("W2 =", np.round(W2.ravel(), 4))
print()
print("Total de parámetros entrenables:", W1.size + b1.size + W2.size + b2.size)

W1: (3, 4) -> conecta 3 entradas con 4 neuronas ocultas
b1: (1, 4)
W2: (4, 1) -> conecta 4 neuronas ocultas con 1 salida
b2: (1, 1)

W1 =
 [[ 0.4056 -0.1129  0.5288  1.2435]
 [-0.1912 -0.1912  1.2894  0.6266]
 [-0.3833  0.443  -0.3784 -0.3803]]
W2 = [ 0.1711 -1.3529 -1.2197 -0.3976]

Total de parámetros entrenables: 21


### 6.1 *Forward propagation* multicapa

La información avanza capa por capa: $Z_1=XW_1+b_1$, $A_1=\mathrm{{ReLU}}(Z_1)$,
$Z_2=A_1W_2+b_2$, $A_2=\sigma(Z_2)$.

In [11]:
y_col = y_train.reshape(-1, 1)

Z1 = X_train @ W1 + b1
A1 = relu(Z1)
Z2 = A1 @ W2 + b2
A2 = sigmoid(Z2)

print("Activaciones de la capa oculta A1 (forma", A1.shape, "):")
print(np.round(A1[:6], 4))
print()
print("Neuronas 'muertas' (siempre en 0) en la capa oculta:",
      int((A1.max(axis=0) == 0).sum()), "de 4")
print()
print("Primeras 5 probabilidades de salida:", np.round(A2.ravel()[:5], 4))
print("Etiquetas reales                   :", y_train[:5])
print()
print("Pérdida inicial (BCE):", round(float(calcular_error(y_col, A2)), 4))

Activaciones de la capa oculta A1 (forma (16, 4) ):
[[0.9874 0.     0.     1.1429]
 [1.0559 0.     0.     1.3316]
 [0.     0.3724 0.0727 0.    ]
 [0.     0.3676 0.1296 0.    ]
 [0.     0.3623 0.3955 0.    ]
 [0.     0.3522 0.7183 0.    ]]

Neuronas 'muertas' (siempre en 0) en la capa oculta: 0 de 4

Primeras 5 probabilidades de salida: [0.4291 0.4137 0.3561 0.3418 0.2744]
Etiquetas reales                   : [0 0 1 1 1]

Pérdida inicial (BCE): 0.9529


### 6.2 *Backpropagation* multicapa, paso a paso

La señal de corrección nace en la salida y **se propaga hacia atrás** atravesando cada capa. El término
`dA1 * derivada_relu(Z1)` es el que "apaga" el gradiente de las neuronas cuyo valor previo fue negativo.

In [12]:
m = len(X_train)

# ---- Forward ----
Z1 = X_train @ W1 + b1
A1 = relu(Z1)
Z2 = A1 @ W2 + b2
A2 = sigmoid(Z2)
error = calcular_error(y_col, A2)

# ---- Capa de salida ----
dZ2 = A2 - y_col
dW2 = A1.T @ dZ2 / m
db2 = np.mean(dZ2, axis=0, keepdims=True)

# ---- Propagación hacia la capa oculta ----
dA1 = dZ2 @ W2.T                       # el error se reparte según W2
dZ1 = dA1 * derivada_relu(Z1)          # se aplica la derivada de ReLU
dW1 = X_train.T @ dZ1 / m
db1 = np.mean(dZ1, axis=0, keepdims=True)

print("Pérdida actual                  :", round(float(error), 6))
print()
print("dZ2 (primeros 5)                :", np.round(dZ2.ravel()[:5], 5))
print("dW2 (gradientes capa de salida) :", np.round(dW2.ravel(), 6))
print("db2                             :", np.round(db2.ravel(), 6))
print()
print("dA1 (error que llega a la capa oculta), primeras 3 filas:\n", np.round(dA1[:3], 6))
print("dZ1 tras multiplicar por ReLU'(Z1), primeras 3 filas:\n", np.round(dZ1[:3], 6))
print("dW1 (gradientes capa oculta):\n", np.round(dW1, 6))
print("db1                             :", np.round(db1.ravel(), 6))
print()
print("Norma del gradiente |dW1| =", round(float(np.linalg.norm(dW1)), 6))
print("Norma del gradiente |dW2| =", round(float(np.linalg.norm(dW2)), 6))

Pérdida actual                  : 0.952865

dZ2 (primeros 5)                : [ 0.42911  0.41368 -0.64392 -0.65825 -0.72563]
dW2 (gradientes capa de salida) : [ 0.208053 -0.128823 -0.153272  0.188029]
db2                             : [-0.13912]

dA1 (error que llega a la capa oculta), primeras 3 filas:
 [[ 0.073418 -0.580546 -0.523391 -0.170615]
 [ 0.070777 -0.559661 -0.504563 -0.164477]
 [-0.110171  0.871161  0.785396  0.256023]]
dZ1 tras multiplicar por ReLU'(Z1), primeras 3 filas:
 [[ 0.073418 -0.       -0.       -0.170615]
 [ 0.070777 -0.       -0.       -0.164477]
 [-0.        0.871161  0.785396  0.      ]]
dW1 (gradientes capa oculta):
 [[ 0.033361 -0.450459 -0.450928 -0.074916]
 [-0.03828   0.492116  0.465746  0.074185]
 [-0.038474  0.490994  0.462845  0.07385 ]]
db1                             : [ 0.038173  0.490061  0.416972 -0.075493]

Norma del gradiente |dW1| = 1.158135
Norma del gradiente |dW2| = 0.34457


### 6.3 Entrenamiento completo (3000 épocas)

Se usa un *learning rate* menor (0.05) que en la red de una capa: con más parámetros, pasos demasiado
grandes pueden hacer oscilar la pérdida. El **orden de actualización es indiferente** porque todos los
gradientes se calcularon con los mismos parámetros de la época.

In [13]:
np.random.seed(42)
W1, b1, W2, b2 = inicializar_parametros(3, 4, 1, semilla=42)
learning_rate = 0.05
epochs = 3000
historial_relu = []

for epoch in range(epochs):
    # 1. Forward propagation
    Z1 = X_train @ W1 + b1
    A1 = relu(Z1)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)

    # 2. Loss
    error = float(calcular_error(y_col, A2))
    historial_relu.append(error)

    # 3. Backpropagation
    m = len(X_train)
    dZ2 = A2 - y_col
    dW2 = A1.T @ dZ2 / m
    db2 = np.mean(dZ2, axis=0, keepdims=True)

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * derivada_relu(Z1)
    dW1 = X_train.T @ dZ1 / m
    db1 = np.mean(dZ1, axis=0, keepdims=True)

    # 4. Actualización (descenso de gradiente)
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    if epoch % 500 == 0:
        print(f"Época {epoch:>5} | Loss {error:.6f}")

print(f"Época {epochs-1:>5} | Loss {historial_relu[-1]:.6f}")

W1_relu, b1_relu, W2_relu, b2_relu = W1.copy(), b1.copy(), W2.copy(), b2.copy()
print()
print("W1 final =\n", np.round(W1_relu, 4))
print("W2 final =", np.round(W2_relu.ravel(), 4))

Época     0 | Loss 0.952865
Época   500 | Loss 0.055464
Época  1000 | Loss 0.025895
Época  1500 | Loss 0.016626
Época  2000 | Loss 0.012180
Época  2500 | Loss 0.009587
Época  2999 | Loss 0.007896

W1 final =
 [[ 0.4751 -0.017   1.174   1.652 ]
 [-0.3647 -0.2951  0.205  -0.1614]
 [-0.562   0.3391 -1.4779 -1.1871]]
W2 final = [-0.6199 -1.3377 -1.8684 -1.6722]


### 6.4 Evaluación de la red multicapa

In [14]:
def forward_multicapa(X_datos, W1, b1, W2, b2, act_oculta=relu):
    """Propagación hacia adelante de la red 3 -> 4 -> 1."""
    Z1 = X_datos @ W1 + b1
    A1 = act_oculta(Z1)
    Z2 = A1 @ W2 + b2
    return sigmoid(Z2), A1, Z1


probs_multi, A1_test, Z1_test = forward_multicapa(X_test, W1_relu, b1_relu, W2_relu, b2_relu)
pred_multi = (probs_multi >= 0.5).astype(int).ravel()
acc_multi = accuracy_score(y_test, pred_multi)

print("=== RED MULTICAPA 3 -> 4 -> 1 (ReLU + Sigmoide) ===")
print(f"{'Fruta':>6} {'P(banano)':>10} {'Predicción':>12} {'Real':>12} {'Resultado':>10}")
for i in range(len(X_test)):
    pred = "BANANO" if pred_multi[i] == 1 else "MANZANA"
    real = "BANANO" if y_test[i] == 1 else "MANZANA"
    ok = "ACIERTO" if pred == real else "ERROR"
    print(f"{i+1:>6} {probs_multi[i,0]:>10.6f} {pred:>12} {real:>12} {ok:>10}")

print()
print("Accuracy en prueba:", round(acc_multi, 4))
print("Matriz de confusión:\n", confusion_matrix(y_test, pred_multi))
print()
print("Activaciones de la capa oculta en prueba (qué neurona responde a cada fruta):")
print(np.round(A1_test, 4))

=== RED MULTICAPA 3 -> 4 -> 1 (ReLU + Sigmoide) ===
 Fruta  P(banano)   Predicción         Real  Resultado
     1   0.014244      MANZANA      MANZANA    ACIERTO
     2   0.000021      MANZANA      MANZANA    ACIERTO
     3   0.985087       BANANO       BANANO    ACIERTO
     4   0.985087       BANANO       BANANO    ACIERTO

Accuracy en prueba: 1.0
Matriz de confusión:
 [[2 0]
 [0 2]]

Activaciones de la capa oculta en prueba (qué neurona responde a cada fruta):
[[1.2102 0.     2.1271 2.2145]
 [1.6851 0.     3.5213 4.3988]
 [0.     0.     0.     0.    ]
 [0.     0.     0.     0.    ]]


---

# 7. Experimento comparativo: ReLU vs Sigmoide en la capa oculta

Ambas redes tienen la **misma arquitectura, los mismos datos, la misma semilla y las mismas épocas**.
Solo cambia la activación de la capa oculta. El objetivo es observar el **desvanecimiento del gradiente**:
como $\sigma'(z)\le 0.25$, cada capa multiplica la señal de corrección por un factor pequeño, y el
aprendizaje se vuelve más lento.

In [15]:
def entrenar_multicapa(act_oculta, derivada_act, learning_rate, epochs, semilla=42):
    """Entrena la red 3->4->1 con la activación indicada y devuelve historial y parámetros."""
    W1, b1, W2, b2 = inicializar_parametros(3, 4, 1, semilla=semilla)
    historial = []
    for _ in range(epochs):
        Z1 = X_train @ W1 + b1
        A1 = act_oculta(Z1)
        Z2 = A1 @ W2 + b2
        A2 = sigmoid(Z2)
        historial.append(float(calcular_error(y_col, A2)))

        m = len(X_train)
        dZ2 = A2 - y_col
        dW2 = A1.T @ dZ2 / m
        db2 = np.mean(dZ2, axis=0, keepdims=True)
        dA1 = dZ2 @ W2.T
        dZ1 = dA1 * derivada_act(Z1)
        dW1 = X_train.T @ dZ1 / m
        db1 = np.mean(dZ1, axis=0, keepdims=True)

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
    return historial, (W1, b1, W2, b2)


def derivada_sigmoid(z):
    s = sigmoid(z)
    return s * (1 - s)


hist_relu, params_relu = entrenar_multicapa(relu, derivada_relu, 0.05, 3000)
hist_sig, params_sig = entrenar_multicapa(sigmoid, derivada_sigmoid, 0.05, 3000)

print(f"{'Época':>7} {'Loss ReLU':>12} {'Loss Sigmoide':>15}")
for e in [0, 500, 1000, 1500, 2000, 2500, 2999]:
    print(f"{e:>7} {hist_relu[e]:>12.6f} {hist_sig[e]:>15.6f}")

print()
print("Pérdida final  ReLU    :", round(hist_relu[-1], 6))
print("Pérdida final  Sigmoide:", round(hist_sig[-1], 6))

  Época    Loss ReLU   Loss Sigmoide
      0     0.952865        1.022297
    500     0.055464        0.114070
   1000     0.025895        0.050687
   1500     0.016626        0.029897
   2000     0.012180        0.020223
   2500     0.009587        0.014925
   2999     0.007896        0.011692

Pérdida final  ReLU    : 0.007896
Pérdida final  Sigmoide: 0.011692


In [16]:
# Norma del gradiente en la capa oculta al inicio del entrenamiento (misma semilla)
W1i, b1i, W2i, b2i = inicializar_parametros(3, 4, 1, semilla=42)
Z1 = X_train @ W1i + b1i
A1r, A1s = relu(Z1), sigmoid(Z1)
Z2r, Z2s = A1r @ W2i + b2i, A1s @ W2i + b2i
dZ2 = sigmoid(Z2r) - y_col
dA1 = dZ2 @ W2i.T

norma_relu = float(np.linalg.norm(X_train.T @ (dA1 * derivada_relu(Z1)) / len(X_train)))
norma_sig = float(np.linalg.norm(X_train.T @ (dA1 * derivada_sigmoid(Z1)) / len(X_train)))

print("Norma ||dW1|| con ReLU en la capa oculta    :", f"{norma_relu:.6e}")
print("Norma ||dW1|| con Sigmoide en la capa oculta:", f"{norma_sig:.6e}")
print("Razón ReLU / Sigmoide                       :", f"{norma_relu / norma_sig:.2f}x")

Norma ||dW1|| con ReLU en la capa oculta    : 1.158135e+00
Norma ||dW1|| con Sigmoide en la capa oculta: 4.375875e-01
Razón ReLU / Sigmoide                       : 2.65x


In [17]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))

axes[0].plot(hist_relu, label="Capa oculta ReLU", color="#27ae60", lw=2)
axes[0].plot(hist_sig, label="Capa oculta Sigmoide", color="#8e44ad", lw=2)
axes[0].plot(historial_1capa, label="Red de una capa (Sigmoide)", color="#2980b9", lw=1.6, ls="--")
axes[0].set_yscale("log")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("Loss (escala log)")
axes[0].set_title("Figura 5. Comparación de curvas de aprendizaje")
axes[0].legend(fontsize=8)

# Fronteras de decisión de los tres modelos
media, desv = scaler.mean_, scaler.scale_
i_f, j_f = 1, 2
g1 = np.linspace(X[:, i_f].min() - 1, X[:, i_f].max() + 1, 300)
g2 = np.linspace(X[:, j_f].min() - 0.1, X[:, j_f].max() + 0.1, 300)
G1, G2 = np.meshgrid(g1, g2)
pts = np.column_stack([np.full(G1.size, media[0]), G1.ravel(), G2.ravel()])
pts_e = (pts - media) / desv

modelos = [("Una capa (Sigmoide)", lambda Xe: sigmoid(Xe @ W_1capa + b_1capa).ravel()),
           ("Multicapa ReLU", lambda Xe: forward_multicapa(Xe, *params_relu)[0].ravel()),
           ("Multicapa Sigmoide", lambda Xe: forward_multicapa(Xe, *params_sig, act_oculta=sigmoid)[0].ravel())]

ax = axes[1]
for nombre, fn in modelos:
    Zg = fn(pts_e).reshape(G1.shape)
    ax.contour(G1, G2, Zg, levels=[0.5], linewidths=2)
for clase, nombre, c in [(0, "Manzana", "#c0392b"), (1, "Banano", "#f1c40f")]:
    msk = y == clase
    ax.scatter(X[msk, i_f], X[msk, j_f], c=c, edgecolors="k", s=60, label=nombre)
ax.set_xlabel("Longitud (cm)"); ax.set_ylabel("Nivel de amarillo")
ax.set_title("Figura 6. Fronteras de decisión (curvas = umbral 0.5)")
ax.legend(fontsize=8)

plt.tight_layout(); plt.savefig("fig04_comparacion_activaciones.png", bbox_inches="tight"); plt.show()

### 7.1 Tabla comparativa de los tres modelos implementados desde cero

Todos comparten dataset, partición, escalado y semilla.

In [18]:
def evaluar_modelo(nombre, fn_prob):
    probs = fn_prob(X_test).ravel()
    preds = (probs >= 0.5).astype(int)
    acc = accuracy_score(y_test, preds)
    loss_test = float(calcular_error(y_test.reshape(-1, 1), probs.reshape(-1, 1)))
    return nombre, loss_test, acc


resultados = []
resultados.append(evaluar_modelo("Una capa · Sigmoide", lambda Xe: sigmoid(Xe @ W_1capa + b_1capa)))
resultados.append(evaluar_modelo("Multicapa · ReLU oculta", lambda Xe: forward_multicapa(Xe, *params_relu)[0]))
resultados.append(evaluar_modelo("Multicapa · Sigmoide oculta", lambda Xe: forward_multicapa(Xe, *params_sig, act_oculta=sigmoid)[0]))

print(f"{'Modelo':<32}{'Loss prueba':>13}{'Accuracy':>11}")
print("-" * 56)
for nombre, loss_t, acc in resultados:
    print(f"{nombre:<32}{loss_t:>13.6f}{acc:>11.4f}")

Modelo                            Loss prueba   Accuracy
--------------------------------------------------------
Una capa · Sigmoide                  0.002713     1.0000
Multicapa · ReLU oculta              0.011104     1.0000
Multicapa · Sigmoide oculta          0.013006     1.0000


## 8. Análisis de resultados

- La **red de una capa** con Sigmoide y BCE alcanza una pérdida muy baja y clasificación correcta en la
  prueba, confirmando que el descenso de gradiente con *backpropagation* converge sobre datos separables.
- La **red multicapa con ReLU** converge más rápido en las primeras épocas que su equivalente con
  Sigmoide, y mantiene una norma de gradiente mayor en la primera capa: evidencia directa del
  desvanecimiento del gradiente en activaciones saturantes.
- Las **fronteras de decisión** son prácticamente equivalentes en este dataset porque el problema es
  linealmente separable; la ventaja de la profundidad se manifiesta en problemas no lineales
  (por ejemplo, MNIST, donde se necesita combinar cientos de píxeles).
- Un *accuracy* de 1.0 sobre **cuatro** frutas de prueba no demuestra que el modelo sea perfecto: el
  conjunto es pequeño y deliberadamente separable con fines didácticos. La generalización real se
  valida en el Notebook 05 con 10 000 imágenes de MNIST.

## 9. Conclusiones del notebook

1. *Backpropagation* no es un algoritmo de aprendizaje en sí mismo: es el procedimiento eficiente para
   obtener los gradientes que luego usa el descenso de gradiente.
2. La derivada de la Sigmoide puede expresarse a partir de su propia salida, lo que hace el cálculo barato.
3. ReLU es la activación estándar en capas ocultas: es barata de evaluar, no se satura en la región
   positiva y produce gradientes de magnitud 1.
4. La Sigmoide se reserva para la **capa de salida binaria**, donde su rango (0,1) se interpreta como probabilidad.
5. Inicializar bien los pesos (He para ReLU) es tan importante como la arquitectura: condiciona la
   magnitud de los gradientes desde la época 0.

## 10. Referencias

- Trask, A. (2019). *Grokking Deep Learning*. Manning Publications.
- Stevens, E., Antiga, L., & Viehmann, T. (2020). *Deep Learning with PyTorch*. Manning Publications.
- Gulli, A., Kapoor, A., Pal, S., & Ciraci, S. (2023). *Deep Learning with TensorFlow and Keras* (3rd ed.). Packt Publishing.